# 01 — Validação da base

**Objetivo:** antes de qualquer análise, conferir se a base reproduz os
números que a ANAC publica. Se não bater, não dá pra avançar: é preciso
saber se a diferença é de escopo, de revisão ou erro meu. Nesta ordem:

1. Os totais: passageiros, ocupação, voos e pontualidade, contra o
   Anuário do Transporte Aéreo e os comunicados mensais da ANAC.
2. A tarifa média e o yield, contra os comunicados de tarifas e o Anuário.
3. Um número da imprensa: a tarifa média de cada empresa em ago/2026.

Faz parte da [etapa 4](../README.md#etapas) do projeto. Os dados vêm de
`dados/marts/validacao.parquet`, que o `construir.py` monta a partir da
coleta de 13/10/2026: um registro por mês e empresa, só com somas e
contagens, pra toda média sair ponderada da razão entre as somas.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ivadivad/voos-em-numeros/main/dados/marts/validacao.parquet"
mart = pd.read_parquet(URL)
mart["mes"] = pd.to_datetime(mart["mes"])
mart["ano"] = mart["mes"].dt.year
print(f"{len(mart)} linhas, {mart['mes'].nunique()} meses "
      f"({mart['mes'].min():%m/%Y} a {mart['mes'].max():%m/%Y}), {mart['empresa'].nunique()} empresas")

1188 linhas, 92 meses (01/2019 a 08/2026), 24 empresas


## 1. Passageiros, ocupação e pontualidade

O que a ANAC publicou, nas palavras dela:

> "No mercado doméstico de passageiros, foram transportados 101 milhões de
> passageiros em 2025, aumento de 8,4% em relação a 2024. A taxa de ocupação
> das aeronaves atingiu 83,6% no mesmo período." — [notícia do Anuário do
> Transporte Aéreo, 07/07/2026](https://www.gov.br/anac/pt-br/noticias/2026/confira-a-nova-edicao-do-anuario-do-transporte-aereo-da-anac)

> "No mercado doméstico, foram transportados 95,3 milhões de passageiros
> pagos no ano passado, 1,7% a mais que o verificado em 2018." —
> [ANAC Informa de janeiro de 2020](https://www.gov.br/anac/pt-br/noticias/newsletter/2020/anac_informa_janeiro_2020.pdf), sobre 2019

Os passageiros vêm da base estatística, a mesma fonte do Anuário. A
`stg_estatisticas` traz os três grupos de voo (regular, não regular e
improdutivo), e a conta abaixo soma os três.

In [2]:
pax = mart.groupby("ano")["passageiros_pagos"].sum() / 1e6

def compara(medida, base, publicado):
    print(f"{medida:30} base {base:8.2f}   publicado {publicado:7.2f}   "
          f"diferença {base - publicado:+6.2f} ({base / publicado - 1:+.2%})")

compara("passageiros 2019 (milhões)", pax[2019], 95.3)
compara("passageiros 2025 (milhões)", pax[2025], 101)
compara("variação 2025/2024 (%)", (pax[2025] / pax[2024] - 1) * 100, 8.4)

a = mart[mart["ano"] == 2025].sum(numeric_only=True)
compara("ocupação 2025 (%)", a.rpk / a.ask * 100, 83.6)

passageiros 2019 (milhões)     base    95.06   publicado   95.30   diferença  -0.24 (-0.25%)
passageiros 2025 (milhões)     base   101.42   publicado  101.00   diferença  +0.42 (+0.41%)
variação 2025/2024 (%)         base     8.44   publicado    8.40   diferença  +0.04 (+0.42%)
ocupação 2025 (%)              base    83.54   publicado   83.60   diferença  -0.06 (-0.07%)


**2025 bate.** 101,42 milhões contra os "101 milhões" arredondados, e
a variação sobre 2024 sai 8,44%, contra 8,4%. A ocupação, RPK sobre ASK,
fica 0,06 ponto abaixo dos 83,6%.

**2019 fica 0,25% abaixo**, 240 mil passageiros num ano de 95 milhões.
O boletim foi escrito em jan/2020, com os dados de então, e a ANAC
republica a série (o `DICIONARIO.md` registra arquivos substituídos e o
acumulado regerado a cada coleta). O mesmo cálculo bater em 2025, com
número publicado há três meses, aponta pra revisão do dado, e não pra
escopo diferente: escopo erraria nos dois anos.

### Os comunicados mensais

> "9,2 milhões no segmento doméstico (1,9% acima do registrado em julho de
> 2025)" — [ANAC, 20/08/2026](https://www.gov.br/anac/pt-br/noticias/2026/setor-aereo-movimenta-11-9-milhoes-de-passageiros-no-pais-no-mes-de-julho)

> "foram 8,9 milhões de passageiros (crescimento de 2,3%) no segmento
> doméstico" — [ANAC, 21/09/2026](https://www.gov.br/anac/pt-br/noticias/2026/aviacao-civil-brasileira-movimentou-11-5-milhoes-de-passageiros-em-agosto), sobre agosto

In [3]:
mensal = mart.groupby("mes")[["passageiros_pagos", "passageiros_pagos_regular"]].sum() / 1e6
for mes, ano_antes, publicado, variacao in [("2026-07", "2025-07", 9.2, 1.9), ("2026-08", "2025-08", 8.9, 2.3)]:
    atual, antes = mensal.loc[pd.Timestamp(mes)], mensal.loc[pd.Timestamp(ano_antes)]
    compara(f"passageiros {mes} (milhões)", atual.passageiros_pagos, publicado)
    compara(f"variação {mes} (%)", (atual.passageiros_pagos / antes.passageiros_pagos - 1) * 100, variacao)
    compara(f"variação {mes}, só regular (%)",
            (atual.passageiros_pagos_regular / antes.passageiros_pagos_regular - 1) * 100, variacao)

passageiros 2026-07 (milhões)  base     9.21   publicado    9.20   diferença  +0.01 (+0.12%)
variação 2026-07 (%)           base     1.82   publicado    1.90   diferença  -0.08 (-4.27%)
variação 2026-07, só regular (%) base     3.87   publicado    1.90   diferença  +1.97 (+103.77%)
passageiros 2026-08 (milhões)  base     8.94   publicado    8.90   diferença  +0.04 (+0.46%)
variação 2026-08 (%)           base     2.12   publicado    2.30   diferença  -0.18 (-7.71%)
variação 2026-08, só regular (%) base     3.16   publicado    2.30   diferença  +0.86 (+37.28%)


**O volume do mês bate; a variação fica 0,1 a 0,2 ponto abaixo.** Julho
dá 9,21 milhões e agosto 8,94, os dois arredondando pro publicado. Já o
crescimento sobre o mesmo mês de 2025 sai 1,8% e 2,1%, contra 1,9% e
2,3%. Contar só os voos regulares não explica (afasta mais: 3,9% e
3,2%), e o comunicado não traz o número de 2025 pra conferir. Fica como
diferença registrada, pequena demais pra mudar qualquer conclusão, e não
resolvida.

### Pontualidade, do VRA

> "No mercado doméstico, em 2025, 7,1% dos voos registraram atraso superior
> a 30 minutos na chegada, 2,7% tiveram atraso superior a 60 minutos e 1,8%
> das operações foram canceladas." — [notícia do Anuário](https://www.gov.br/anac/pt-br/noticias/2026/confira-a-nova-edicao-do-anuario-do-transporte-aereo-da-anac)

O comunicado não diz o denominador. A conta usa os dois candidatos: os
voos realizados e todos os voos (realizados, cancelados e com situação não
informada).

In [4]:
for minutos, publicado in [(30, 7.1), (60, 2.7)]:
    atrasados = a[f"voos_atraso_{minutos}"]
    compara(f"atraso > {minutos} min / realizados (%)", atrasados / a.voos_realizados * 100, publicado)
    compara(f"atraso > {minutos} min / todos (%)", atrasados / a.voos * 100, publicado)
compara("cancelados / todos (%)", a.voos_cancelados / a.voos * 100, 1.8)

atraso > 30 min / realizados (%) base     7.12   publicado    7.10   diferença  +0.02 (+0.28%)
atraso > 30 min / todos (%)    base     7.00   publicado    7.10   diferença  -0.10 (-1.45%)
atraso > 60 min / realizados (%) base     2.77   publicado    2.70   diferença  +0.07 (+2.71%)
atraso > 60 min / todos (%)    base     2.73   publicado    2.70   diferença  +0.03 (+0.94%)
cancelados / todos (%)         base     1.72   publicado    1.80   diferença  -0.08 (-4.30%)


**Os três batem dentro de 0,1 ponto, mas nenhum denominador reproduz os
dois atrasos ao mesmo tempo.** Sobre os realizados, o atraso de 30 min
dá 7,12% (o 7,1% publicado) e o de 60 min, 2,77%. Sobre todos os voos, o
de 60 min dá 2,73% (o 2,7%) e o de 30 min, 7,00%. Os cancelados ficam em
1,72%, contra 1,8%.

A diferença cabe nos filtros que a staging aplica e que a ANAC pode não
aplicar (os voos com tipo de linha `NA`, todos cancelados, ficam de fora
aqui; ver `DICIONARIO.md`). O VRA, que é a base da pontualidade da etapa
9, reproduz o publicado na casa do décimo de ponto.

## 2. Tarifa média e yield

A ANAC publica a tarifa média todo mês. Os três comunicados mais
recentes, nas palavras dela:

> "No mês, a tarifa real média da passagem aérea no país, considerando
> todas as rotas, foi de R\$ 660,54 por trecho" — [junho](https://www.gov.br/anac/pt-br/noticias/2026/anac-publica-dados-tarifarios-do-mes-de-junho)

> "Em julho, a tarifa real média da passagem aérea no país, considerando
> todas as rotas, foi de R\$ 703,52 por trecho" — [julho](https://www.gov.br/anac/pt-br/noticias/2026/o-valor-da-tarifa-media-foi-de-r-703)

> "No mês, a tarifa real média constatada no país, considerando todas as
> rotas e todas as empresas, foi de R\$ 776,87 por trecho" — [agosto](https://www.gov.br/anac/pt-br/noticias/2026/dados-de-tarifas-aereas-do-mes-de-agostoestao-disponiveis-para-consulta)

A conta candidata é a média ponderada pelos assentos: a receita (tarifa
vezes assentos vendidos a ela) dividida pelos assentos, no mês da venda.

In [5]:
tarifa = mart.groupby("mes")[["receita_tarifas", "assentos_vendidos"]].sum()
tarifa["media"] = tarifa["receita_tarifas"] / tarifa["assentos_vendidos"]

for mes, publicado in [("2026-06", 660.54), ("2026-07", 703.52), ("2026-08", 776.87)]:
    compara(f"tarifa média {mes} (R$)", tarifa.loc[pd.Timestamp(mes), "media"], publicado)

tarifa média 2026-06 (R$)      base   660.54   publicado  660.54   diferença  -0.00 (-0.00%)
tarifa média 2026-07 (R$)      base   703.52   publicado  703.52   diferença  +0.00 (+0.00%)
tarifa média 2026-08 (R$)      base   776.87   publicado  776.87   diferença  -0.00 (-0.00%)


**Bate no centavo, nos três meses.** A fórmula está confirmada: média
ponderada pelos assentos, no mês da venda. E o "real" do comunicado é
o valor do próprio mês: o IPCA de referência é o do mês publicado, então
o último mês sai nominal.

### As faixas de preço

Os mesmos comunicados dão a parcela dos assentos abaixo de R\$ 500 e acima
de R\$ 1,5 mil: 47,2% e 6,3% em junho, 45,4% e 8,3% em julho, 34,8% e 9,2%
em agosto. O de agosto, por exemplo: "34,8% dos assentos comercializados
no mês se situou na faixa abaixo de R\$ 500, sendo que 9,2% dos assentos
foram vendidos por mais de R\$ 1,5 mil". O mart conta "até R\$ 500", com
o próprio R\$ 500 dentro.

In [6]:
faixas = mart.groupby("mes")[["assentos_ate_500", "assentos_acima_1500", "assentos_vendidos"]].sum()
for mes, ate_500, acima_1500 in [("2026-06", 47.2, 6.3), ("2026-07", 45.4, 8.3), ("2026-08", 34.8, 9.2)]:
    f = faixas.loc[pd.Timestamp(mes)]
    compara(f"até R$ 500, {mes} (%)", f.assentos_ate_500 / f.assentos_vendidos * 100, ate_500)
    compara(f"acima de R$ 1.500, {mes} (%)", f.assentos_acima_1500 / f.assentos_vendidos * 100, acima_1500)

até R$ 500, 2026-06 (%)        base    47.17   publicado   47.20   diferença  -0.03 (-0.07%)
acima de R$ 1.500, 2026-06 (%) base     6.31   publicado    6.30   diferença  +0.01 (+0.21%)
até R$ 500, 2026-07 (%)        base    45.44   publicado   45.40   diferença  +0.04 (+0.08%)
acima de R$ 1.500, 2026-07 (%) base     8.30   publicado    8.30   diferença  +0.00 (+0.04%)
até R$ 500, 2026-08 (%)        base    34.77   publicado   34.80   diferença  -0.03 (-0.09%)
acima de R$ 1.500, 2026-08 (%) base     9.15   publicado    9.20   diferença  -0.05 (-0.53%)


**As seis parcelas batem na casa decimal**, com o valor da fronteira
dentro da faixa de baixo, como a ANAC conta.

### O valor real: a correção pelo IPCA

Os comunicados comparam o mês com o mesmo mês dos anos anteriores "em
valores atualizados pelo IPCA": aumento de 0,3% sobre jun/2025, queda de
5,2% sobre jul/2025 e aumento de 14,5% sobre ago/2025. Pra reproduzir,
cada mês antigo é levado ao preço do mês publicado pela variação do IPCA
entre os dois. O IPCA mensal vem do Banco Central (série 433 do SGS).

In [7]:
import json
import time
import urllib.request

IPCA = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2019"
pedido = urllib.request.Request(IPCA, headers={"User-Agent": "voos-em-numeros/0.1 (+https://github.com/ivadivad/voos-em-numeros)"})
for tentativa in range(3):  # a API do BC falha de vez em quando: 502, ou erro com status 200
    try:
        with urllib.request.urlopen(pedido, timeout=60) as resposta:
            serie = json.load(resposta)
        if not isinstance(serie, list):  # {"erro": ...} com status 200
            raise ValueError(f"a API devolveu {str(serie)[:80]}")
        ipca = pd.DataFrame(serie)
        break
    except (OSError, ValueError):  # erro HTTP, falha de rede, ou corpo que nao e a serie
        if tentativa == 2:
            raise
        time.sleep(10)
ipca["mes"] = pd.to_datetime(ipca["data"], format="%d/%m/%Y")
ipca["indice"] = (1 + ipca["valor"].astype(float) / 100).cumprod()
indice = ipca.set_index("mes")["indice"]

def real(valor, mes, base):
    """Valor do mes, levado ao preco do mes base pelo IPCA."""
    return valor * indice[pd.Timestamp(base)] / indice[pd.Timestamp(mes)]

for mes, antes, publicado in [("2026-06", "2025-06", 0.3), ("2026-07", "2025-07", -5.2), ("2026-08", "2025-08", 14.5)]:
    agora = tarifa.loc[pd.Timestamp(mes), "media"]
    um_ano_antes = real(tarifa.loc[pd.Timestamp(antes), "media"], antes, mes)
    compara(f"variação real {mes} (%)", (agora / um_ano_antes - 1) * 100, publicado)

variação real 2026-06 (%)      base     0.35   publicado    0.30   diferença  +0.05 (+16.30%)
variação real 2026-07 (%)      base    -5.20   publicado   -5.20   diferença  -0.00 (+0.05%)
variação real 2026-08 (%)      base    14.50   publicado   14.50   diferença  +0.00 (+0.01%)


**As três variações reais batem**: julho e agosto no centésimo, junho a
0,05 ponto. O método fica confirmado: cada mês é corrigido pelo IPCA até
o mês de referência.

### A tarifa do ano, no Anuário

> "Em 2025, a tarifa média foi de aproximadamente R\$ 648,00 por trecho,
> queda de 3,3% em relação a 2024 (R\$ 670,00) e de 10,9% na comparação com
> 2022 (cerca de R\$ 727,00)." — [notícia do Anuário](https://www.gov.br/anac/pt-br/noticias/2026/confira-a-nova-edicao-do-anuario-do-transporte-aereo-da-anac)

O comunicado diz "em valores reais", sem o mês de referência. A conta
abaixo leva a receita de cada mês a dez/2025, o último mês do ano do
Anuário, e divide pelos assentos do ano.

In [8]:
tarifa["receita_real"] = [real(r, mes, "2025-12") for mes, r in tarifa["receita_tarifas"].items()]
tarifa["ano"] = tarifa.index.year
anual = tarifa.groupby("ano")[["receita_real", "assentos_vendidos"]].sum()
anual["media_real"] = anual["receita_real"] / anual["assentos_vendidos"]

for ano, publicado in [(2022, 727), (2024, 670), (2025, 648)]:
    compara(f"tarifa real {ano} (R$ de dez/2025)", anual.loc[ano, "media_real"], publicado)
compara("variação real 2025/2024 (%)", (anual.loc[2025, "media_real"] / anual.loc[2024, "media_real"] - 1) * 100, -3.3)

tarifa real 2022 (R$ de dez/2025) base   727.01   publicado  727.00   diferença  +0.01 (+0.00%)
tarifa real 2024 (R$ de dez/2025) base   670.11   publicado  670.00   diferença  +0.11 (+0.02%)
tarifa real 2025 (R$ de dez/2025) base   649.06   publicado  648.00   diferença  +1.06 (+0.16%)
variação real 2025/2024 (%)    base    -3.14   publicado   -3.30   diferença  +0.16 (-4.80%)


**2022 e 2024 batem no real; 2025 fica R\$ 1 acima.** Com a receita em
preços de dez/2025, 2022 dá R\$ 727,01 e 2024, R\$ 670,11: é essa a base
do Anuário. 2025 dá R\$ 649,06, contra "aproximadamente R\$ 648,00", e a
queda sai 3,1% em vez de 3,3%. Os meses de 2025 são os mais recentes do
Anuário, os mais sujeitos a revisão depois de jul/2026, quando ele saiu;
a diferença de 0,16% é desse tamanho.

### O yield

O yield é o preço do quilômetro. Na definição da ANAC, "representa o
valor médio pago por quilômetro entre a origem e o destino do passageiro,
independentemente de escalas ou conexões" ([Relatório de Tarifas Aéreas
Domésticas do 1º trimestre de 2018](https://static.poder360.com.br/2018/06/Relatorio_1960867_Relatorio_de_Tarifas_Aereas_Domesticas___2018_T1.pdf), cópia no Poder360).
A distância é a em linha reta entre os dois aeroportos, a do
`sql/14_distancia.sql`, e o yield do ano é a receita dividida pelos
assentos-km, só nos pares com coordenada nas duas pontas.

O valor do Anuário não está na notícia da ANAC. Duas fontes trazem:

> "O Yield Doméstico Médio, indicador que mede o valor cobrado por
> quilômetro voado, também recuou 4,9% no período." — [InfoMoney,
> 08/07/2026](https://www.infomoney.com.br/?p=3403609), sobre o Anuário de 2025

O [Panorama de Transportes de dez/2025 da INFRA S.A.](https://ontl.infrasa.gov.br/wp-content/uploads/2026/02/Panorama-Transportes-Dezembro-2025.pdf), com fonte
na ANAC, dá yield médio de 0,50 R\$/km em 2025 e variação de -4,20%
sobre 2024.

In [9]:
km = mart.groupby("mes")[["receita_com_distancia", "assentos_km"]].sum()
km["receita_real"] = [real(r, mes, "2025-12") for mes, r in km["receita_com_distancia"].items()]
km["ano"] = km.index.year
yield_ano = km.groupby("ano")[["receita_real", "assentos_km"]].sum()
yield_ano["yield"] = yield_ano["receita_real"] / yield_ano["assentos_km"]

compara("yield 2025 (R$/km de dez/2025)", yield_ano.loc[2025, "yield"], 0.50)
variacao_yield = (yield_ano.loc[2025, "yield"] / yield_ano.loc[2024, "yield"] - 1) * 100
compara("variação do yield 2025/2024 (%)", variacao_yield, -4.9)
compara("variação do yield, INFRA S.A. (%)", variacao_yield, -4.2)
cobertura = mart.groupby("ano")[["receita_com_distancia", "receita_tarifas"]].sum()
print(f"receita com distância conhecida, 2024: {cobertura.loc[2024].iloc[0] / cobertura.loc[2024].iloc[1]:.2%}")

yield 2025 (R$/km de dez/2025) base     0.50   publicado    0.50   diferença  -0.00 (-0.37%)
variação do yield 2025/2024 (%) base    -4.65   publicado   -4.90   diferença  +0.25 (-5.16%)
variação do yield, INFRA S.A. (%) base    -4.65   publicado   -4.20   diferença  -0.45 (+10.65%)
receita com distância conhecida, 2024: 99.26%


**O yield bate no nível e chega perto na variação.** 2025 dá R\$ 0,50
por km, o valor da INFRA S.A., com 99,26% da receita de 2024 em pares de
distância conhecida. A queda sobre 2024 sai 4,65%, entre os 4,9% do
Anuário e os 4,2% da INFRA S.A. Parte dos 0,25 ponto até o Anuário pode
ser a mesma diferença de 2025 vista na tarifa (que caiu 3,1% aqui e 3,3%
lá); a distância não explica, porque bate com a da ANAC em 3.522 pares
(teste `distancia_bate_com_a_anac.sql`). O valor do Anuário em R\$/km não
está numa fonte que eu tenha conseguido abrir.

## 3. Um número da imprensa: a tarifa de cada empresa

A ANAC não põe a tarifa por empresa nos comunicados, mas o painel dela
tem, e a imprensa repercutiu a de agosto:

> "a companhia registrou a menor tarifa média entre as três maiores aéreas
> do país no mês: R\$ 764,17 por trecho, abaixo da Azul (R\$ 772,98), da Gol
> (R\$ 814,47) e da média geral do mercado (R\$ 776,87)." — [Alta Renda
> Blog, 27/09/2026](https://altarendablog.com.br/2026/09/27/latam-tem-a-menor-tarifa-media-entre-as-grandes-aereas-do-brasil-em-agosto-segundo-a-anac-veja-o-que-isso-muda-para-quem-viaja/)

A companhia é a LATAM, código `TAM` nas bases da ANAC. A mesma reportagem
traz a alta sobre julho (7,06% da LATAM, 9,46% da Azul e 21,6% da Gol) e
avisa que o levantamento foi apresentado pela própria LATAM, "com base no
painel da agência". Aqui, a empresa é a que vendeu a passagem, como no
painel.

In [10]:
por_empresa = mart.groupby(["mes", "empresa"])[["receita_tarifas", "assentos_vendidos"]].sum()
por_empresa["media"] = por_empresa["receita_tarifas"] / por_empresa["assentos_vendidos"]

for empresa, nome, publicado, alta in [("TAM", "LATAM", 764.17, 7.06), ("AZU", "Azul", 772.98, 9.46), ("GLO", "Gol", 814.47, 21.6)]:
    agosto = por_empresa.loc[(pd.Timestamp("2026-08"), empresa), "media"]
    julho = real(por_empresa.loc[(pd.Timestamp("2026-07"), empresa), "media"], "2026-07", "2026-08")
    compara(f"tarifa {nome}, ago/2026 (R$)", agosto, publicado)
    compara(f"alta real {nome} sobre julho (%)", (agosto / julho - 1) * 100, alta)

tarifa LATAM, ago/2026 (R$)    base   764.17   publicado  764.17   diferença  -0.00 (-0.00%)
alta real LATAM sobre julho (%) base     7.06   publicado    7.06   diferença  +0.00 (+0.04%)
tarifa Azul, ago/2026 (R$)     base   772.98   publicado  772.98   diferença  +0.00 (+0.00%)
alta real Azul sobre julho (%) base     9.47   publicado    9.46   diferença  +0.01 (+0.09%)
tarifa Gol, ago/2026 (R$)      base   814.47   publicado  814.47   diferença  -0.00 (-0.00%)
alta real Gol sobre julho (%)  base    21.61   publicado   21.60   diferença  +0.01 (+0.03%)


**Bate no centavo pras três empresas, e as altas sobre julho ficam a no
máximo 0,01 ponto** (a da Azul dá 9,47%, contra 9,46%). As altas estão em
valores reais: julho levado a preços de agosto pelo IPCA, como nos
comunicados da ANAC. Isso confirma, além da fórmula, que a empresa das
tarifas é a que vendeu, a mesma do painel.

## O que ficou validado

| O que | Fonte | Resultado |
|---|---|---|
| Tarifa média mensal | Comunicados da ANAC, jun a ago/2026 | Igual no centavo |
| Tarifa média por empresa | Imprensa, ago/2026 | Igual no centavo |
| Faixas de preço | Comunicados da ANAC | Iguais na casa decimal |
| Variações reais da tarifa | Comunicados da ANAC | A no máximo 0,05 ponto |
| Tarifa real anual | Anuário 2025 | 2022 e 2024 iguais; 2025 R\$ 1 acima |
| Passageiros | Anuário 2025 e comunicados de jul e ago/2026 | Iguais no arredondamento publicado; a alta mensal fica até 0,2 ponto abaixo |
| Ocupação | Anuário 2025 | 0,06 ponto abaixo |
| Pontualidade | Anuário 2025 | Dentro de 0,1 ponto |
| Yield | INFRA S.A. e imprensa | Nível igual; variação 0,25 ponto longe do Anuário |
| Passageiros de 2019 | Boletim de jan/2020 | 0,25% abaixo, provável revisão |

**A base está validada.** Tarifa, passageiros e pontualidade reproduzem
o que a ANAC publica, e a fórmula da tarifa média sai confirmada pelo
centavo, não por aproximação. As diferenças que sobram ficam
registradas na tabela: são pequenas demais pra mudar uma conclusão, e
nenhuma aponta erro de leitura.